# Data Understanding

This notebook loads the EMSCAD (Employment Scam Aegean Dataset) job postings dataset and confirms its basic shape, column cardinality, class balance, missing-value pattern, presence of duplicate records, and a first look at the text and metadata fields. The goal of this notebook is to understand the dataset being used, not to clean or model the data yet.

In [ ]:
# Import pandas and configure it to display every column
import pandas as pd
pd.set_option("display.max_columns", None)

## Load the Dataset

The EMSCAD CSV file is loaded into a dataframe, and its shape is printed. This confirms the number of rows and columns available to work with.

In [ ]:
# Load the job postings dataset and report its shape
df = pd.read_csv("../data/raw/fake_job_postings.csv")
print(df.shape)

The dataset contains 17,880 rows and 18 columns.

## Preview the Data

Knowing the shape alone is not enough to understand the dataset. Seeing the actual row content, the dtype pandas has assigned to each column, and summary statistics for the numeric columns is also important to ensure that later work (text cleaning, metadata handling) is based on real data.

In [ ]:
# Preview the first 5 rows to see actual field content
df.head()

The text fields (`company_profile`, `description`, `requirements`, `benefits`) contain long lines of text, and some words are fused together with no space between them (for example, `What you will get from usThrough being part of` in the `benefits` column). This happens when original HTML tags or line breaks are stripped out without leaving a space behind, and it confirms that text cleaning is necessary. `location` is a single combined string (country, state, city), and `salary_range` contains null values which confirms that missing values need to be handled later on.

In [ ]:
# Check each column's dtype and non-null count in one summary
df.info()

All 18 columns are loaded with the expected dtype: `job_id`, `telecommuting`, `has_company_logo`, `has_questions`, and `fraudulent` are integers, and every other column is text.

In [ ]:
# Summarise the numeric columns
df.describe()

`describe()` only summarises the five numeric columns. `job_id` is just a row identifier and carries no modelling signal. The three binary flag columns are numeric, so their averages only serve as simple rates: only 4.29% of postings are telecommuting, 79.53% have a company logo, and 49.17% include screening questions. The remaining columns contain text.

## Check Column Cardinality

A column that only contains one distinct value carries no information for a model to learn from. Every column is checked to confirm it has more than one unique value before it is trusted as a potential feature.

In [ ]:
# Count the number of unique values per column, sorted from lowest to highest
unique_value_counts = df.nunique().sort_values()
print(unique_value_counts)

# Flag any column with one or fewer unique values as unusable
constant_columns = list(unique_value_counts[unique_value_counts <= 1].index)
print("\nConstant columns:", constant_columns if constant_columns else "None")

Every column has at least two unique values. `has_company_logo`, `has_questions`, `telecommuting`, and `fraudulent` are at the low end with only 2 distinct values, which is expected since they are binary flags or the label itself. No column is constant, so no column needs to be dropped for carrying zero information.

## Check the Class Balance

The exact ratio of fraudulent versus legitimate postings is confirmed to ensure that every later modelling decision (e.g. class weighting, choice of metric) is justified by a real number instead of an assumption.

In [ ]:
# Calculate the proportion of fraudulent versus legitimate postings
fraud_rate = df["fraudulent"].value_counts(normalize=True).apply(lambda x: f"{x * 100:.2f}%")
print(fraud_rate)

Fraudulent postings make up approximately 4.8% of the dataset. This confirms that accuracy alone will be a misleading evaluation metric later on, because a model that always identifies a posting as legitimate will be correct 95% of the time.

## Inspect Missing Values

Columns like `salary_range` are known to contain missing values. The number and percentage of missing values per column are checked to gain insights which can be used to determine how missing values are handled later.

In [ ]:
# Calculate the number of missing values per column, sorted from most to least missing
missing_values_by_column = df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False)
print("Count:")
print(missing_values_by_column)

# Calculate the percentage of missing values per column, sorted from most to least missing
missing_value_percentage_by_column = (df.isna().mean() * 100).round(2)
missing_value_percentage_by_column = missing_value_percentage_by_column[missing_value_percentage_by_column > 0].sort_values(ascending=False).apply(lambda x: f"{x:.2f}%")
print("\nPercentage:")
print(missing_value_percentage_by_column)

`salary_range` is missing in 83.96% of postings, leaving too few real values (only about 16% of postings) to use as a normal feature on its own. `department` (64.58%), `required_education` (45.33%), `benefits` (40.34%), `required_experience` (39.43%), `function` (36.10%), `industry` (27.42%), `employment_type` (19.41%), `company_profile` (18.50%), and `requirements` (15.08%) all have meaningful missing data and will need a placeholder value imputed instead of being dropped. `location` is only missing in 1.94% of postings, and `description`, `title`, `job_id`, `telecommuting`, `has_questions`, `has_company_logo`, and `fraudulent` are effectively complete (0% or near-0% missing).

## Look at the Text Fields

The text fields will be combined and cleaned in `02_data_preparation.ipynb`, so a closer look at how much raw content they actually carry is useful now. The character length of each field is measured (treating a missing value as length 0). One legitimate and one fraudulent posting are printed side by side to see what the raw text looks like before any cleaning is applied.

In [ ]:
# Calculate the character length of each text field, treating a missing value as length 0
text_fields = ["title", "company_profile", "description", "requirements", "benefits"]
text_lengths = df[text_fields].fillna("").apply(lambda column: column.str.len())
text_lengths.describe().round(1)

`description` is the longest field on average (around 1,218 characters) and the most variable (up to nearly 15,000 characters), while `benefits` is the shortest and most often empty (median 45 characters, with a quarter of postings having none at all). This wide spread confirms that combining the text fields together, as planned for `02_data_preparation.ipynb`, needs to tolerate very short and very long postings alike.

In [ ]:
# Print the description of one legitimate and one fraudulent posting to see raw text firsthand
legitimate_example = df.loc[df["fraudulent"] == 0, "description"].dropna().iloc[0]
fraudulent_example = df.loc[df["fraudulent"] == 1, "description"].dropna().iloc[0]

print("Legitimate posting example:\n", legitimate_example[:300])
print("\nFraudulent posting example:\n", fraudulent_example[:300])

Both examples read as ordinary job-description prose, but the fraudulent example contains a raw HTML entity (`&amp;`). The issue of words being fused together has already been noted earlier. However, a single example from each class is not proof of a systematic pattern, but it does confirm that the text cleaning step in `02_data_preparation.ipynb` needs to handle HTML entities and stray characters, not just ordinary punctuation.

## Look at the Metadata Fields

The low-cardinality categorical fields and the binary flag columns are looked at next: their most common categories, and whether the fraud rate visibly differs across their values. The three lowest-cardinality categorical columns from the earlier cardinality check are used here (`employment_type`: 5, `required_experience`: 7, `required_education`: 13 unique values). The next lowest, `function` (37) and `industry` (131), are left out because a value-count table stops being a useful glance at that scale. `department`, `company_profile`, `location`, and `salary_range` are excluded for the same reason as they are at an even larger scale (874–3,105 unique values). At that scale, they behave more like free text or identifiers instead of categories. A field that splits cleanly by fraud rate will be determined as a strong, low-cost signal for the classical model. A field with no visible split is less likely to be worth keeping, especially when it also has a lot of missing values that would need filling in.

In [ ]:
# Compare the fraud rate across each category of the low-cardinality metadata fields,
# alongside how many postings back each rate
categorical_fields = ["employment_type", "required_experience", "required_education"]
print("Fraud rate:")
for field in categorical_fields:
    fraud_rate_by_category = df.groupby(field)["fraudulent"].agg(rate="mean", count="count")
    fraud_rate_by_category = fraud_rate_by_category.sort_values("rate", ascending=False)
    fraud_rate_by_category["rate"] = (fraud_rate_by_category["rate"] * 100).round(2).apply(lambda x: f"{x}%")
    print(fraud_rate_by_category)
    print()

`count` is the total number of postings in each category: fraudulent and legitimate combined. `rate` is the percentage of the total which is fraudulent. `employment_type` shows a real spread: Part-time postings are fraudulent 9.28% of the time versus 0.83% for Temporary, and both categories have hundreds of postings behind them. `required_experience` shows a similar, smaller spread, from 7.09% (Executive) down to 1.83% (Associate). `required_education` shows an extreme outlier: some High School Coursework at 74.07%, but that category only has 27 postings behind it which makes the rate unreliable and likely a small-sample artifact instead of a real signal. Besides that, the rest of `required_education` still shows a meaningful, well-supported spread, from 11.18% (Certification, 170 postings) down to 1.94% (Bachelor's Degree, 5,145 postings).

In [ ]:
# Compare the fraud rate across each binary flag column
binary_fields = ["telecommuting", "has_company_logo", "has_questions"]
for field in binary_fields:
    print((df.groupby(field)["fraudulent"].mean() * 100).round(2).apply(lambda x: f"{x}%"))
    print()

All three binary flags show a visible split: telecommuting postings are fraudulent almost twice as often (8.34% versus 4.69%), postings without a company logo are fraudulent nearly eight times as often (15.93% versus 1.99%), and postings without screening questions are fraudulent more than twice as often (6.78% versus 2.84%). `has_company_logo` in particular looks like a strong signal worth keeping as a feature in the classical model.

## Check for Duplicate Records

Duplicate postings can leak between the train, validation, and test splits later on, letting the model be evaluated on something it has already partly memorised, resulting in the inflation of reported metrics. The dataset is checked for exact full-row duplicates, as well as duplicate `job_id` values, which should be unique by construction.

In [ ]:
# Count exact full-row duplicates and duplicate job_id values
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate job_id values:", df["job_id"].duplicated().sum())

No full-row duplicates and no duplicate `job_id` values were found. This dataset does not need de-duplication before the train/validation/test split.

## Summary

- Confirmed the dataset loads to 17,880 rows and 18 columns.
- Confirmed every column has more than one unique value, so no column is constant and none needs to be dropped for carrying zero information.
- Confirmed the fraud rate is approximately 4.8% (866 fraudulent postings), justifying `class_weight="balanced"` over resampling (e.g. SMOTE) later on. `class_weight="balanced"` penalises misclassifying the minority (fraudulent) class more heavily during training without changing the training data itself, whereas SMOTE generates synthetic fraudulent rows by interpolating in feature space. Since the classical model's features are high-dimensional, sparse TF-IDF vectors, interpolating between them is both computationally expensive and prone to producing synthetic feature vectors that do not correspond to any real, coherent text. This makes `class_weight="balanced"` the more effective choice.
- Confirmed which columns have heavy missing data (`salary_range`, `department`, `required_education`, `benefits`, `required_experience`, `function`, `industry`, `employment_type`, `company_profile`, `requirements`).
- Confirmed the text fields vary widely in length and contain words fused together and stray HTML entities, confirming that both combining and cleaning are needed before modelling.
- Confirmed the binary flag columns (`telecommuting`, `has_company_logo`, `has_questions`) each show a visible fraud-rate split, with `has_company_logo` the strongest and most complete of the three. This is worth keeping as a feature in the classical model.
- Confirmed there are no full-row duplicates or duplicate `job_id` values, so no de-duplication is needed before the train/validation/test split.
- No data was cleaned or transformed in this notebook. That begins in `02_data_preparation.ipynb`.